## Topic 3: Chunking Strategy Comparison

### Concept, Intuition, Why It Exists

- "Chunking" isn't one algorithm — it's a family of strategies that all solve the same two problems from the first topic (context limits, retrieval precision) but make very different trade-offs about *where* to cut.
- **Fixed-size chunking**: cuts every N characters or tokens, with no regard for sentence or word boundaries at all. The simplest possible strategy, and the baseline every other strategy improves on.
- **Sentence-aware chunking (what this project has)**: cuts only at sentence boundaries, accumulating sentences until adding the next one would exceed a size limit. Never breaks a sentence in half, but still has no awareness of whether consecutive sentences are actually about the same topic.
- **Semantic chunking**: goes a step further than sentence-awareness — instead of cutting purely on a size limit, it cuts where the *meaning* shifts. Concretely, this usually means embedding consecutive sentences and watching for a drop in similarity between neighboring sentences; a big drop signals a topic change, which becomes the cut point regardless of accumulated size.
- **Structure-aware chunking (Q&A pairs)**: instead of deriving chunk boundaries from size or meaning-drift, it uses the document's own existing structure — a question and its answer, a numbered SOP step, a table row — as the natural, pre-existing chunk boundary. This is the strongest strategy when the source data already has clean internal structure to exploit, because it sidesteps the size-vs-meaning trade-off entirely: the document tells you where the boundaries are.

### Internal Working

- Fixed-size: track a running character/token count; cut the instant the limit is hit, regardless of what's mid-word or mid-sentence at that point.
- Sentence-aware: split the text into sentences first; accumulate sentences into a chunk until the next one would exceed the size limit, then start a new chunk (with optional overlap, from the previous topic).
- Semantic: split into sentences, embed each one, compute similarity between consecutive sentence embeddings, and cut wherever similarity drops below a threshold — chunk size becomes a secondary constraint (a cap), not the primary cutting signal.
- Structure-aware (Q&A): parse the document's known structure directly — e.g. regex or pattern matching for "Q:" / "A:" markers, or numbered list items — and emit one chunk per structural unit, with no size-based logic involved at all unless a single structural unit is itself too large.

### How It's Implemented In This Project

- The project currently uses sentence-aware chunking (`chunk_text()`, covered in the previous topic) as the general-purpose default for JSON-sourced PDF pages and similar prose content.
- Structure-aware chunking is a natural fit for any FAQ-style source content in this project — wherever a document already presents clean Q&A pairs, splitting on that structure directly outperforms running generic sentence-aware chunking over the same content and hoping size-based cuts happen to land between pairs.

### Real-World Issues, Edge Cases, Debugging

- **Fixed-size chunking's failure mode is visible immediately**: cutting mid-word or mid-sentence produces chunks that are syntactically broken, which both reads badly if ever shown to a user and embeds poorly, since the embedding model is processing a fragment, not a coherent unit.
- **Sentence-aware chunking's failure mode is subtler**: it guarantees sentence integrity but not topical integrity — a chunk can still legally contain three unrelated sentences just because they happened to fit under the size limit together, producing the same "blurry embedding" problem chunking was meant to solve in the first place, just less severely than fixed-size.
- **Semantic chunking's failure mode is cost and instability**: it requires embedding every sentence just to *decide* where to cut, before the real per-chunk embedding even happens — meaningfully more compute per document than sentence-aware chunking. The similarity-drop threshold is also a tuning problem with the same sensitivity issues as the near-duplicate threshold from an earlier topic — too sensitive, and it over-splits into too many tiny chunks; too lenient, and it barely improves on plain sentence-aware chunking.
- **Structure-aware chunking's failure mode is brittleness**: it only works as well as the structure-detection logic. A Q&A document with inconsistent formatting (some pairs marked "Q:"/"A:", others just formatted as bold text with no marker) will have some pairs correctly extracted and others silently missed or merged incorrectly — this strategy trades generality for precision, and breaks quietly when the assumed structure doesn't hold.

### Design Decisions & Trade-offs

- Sentence-aware as the project's default general-purpose strategy: meaningfully better than fixed-size for near-zero extra cost (just sentence splitting, no embeddings needed to decide cut points), even though it doesn't fully solve topical coherence the way semantic chunking would.
- Reserving structure-aware chunking specifically for content that has known, reliable internal structure (Q&A pairs), rather than trying to force one universal strategy across every source type — this mirrors the same "right granularity for the right source" principle used at the loader level for text/CSV/JSON.
- Semantic chunking deliberately not adopted as the default here: the added embedding cost and threshold-tuning complexity isn't justified unless sentence-aware chunking is measurably failing on this corpus — a trade-off worth revisiting if retrieval quality issues are specifically traced back to topically-mixed chunks.

### Alternatives & When To Use Each

- Fixed-size chunking — only when speed/simplicity matters far more than quality, or as a quick baseline to compare smarter strategies against.
- Sentence-aware chunking — solid general-purpose default for prose content (policy text, SOPs, FAQ answers as paragraphs) where no embeddings-at-chunk-time cost is wanted; this project's current default.
- Semantic chunking — worth the extra embedding cost when sentence-aware chunking is confirmed (not assumed) to be producing topically-mixed chunks that hurt retrieval precision, typically on long-form prose with frequent unmarked topic shifts.
- Structure-aware chunking — the best choice whenever source content already has reliable, consistent internal structure (Q&A pairs, numbered SOP steps, table rows) to exploit directly, since it sidesteps the size-vs-meaning trade-off entirely.

### Common Mistakes & Production Failures

- Defaulting to fixed-size chunking purely because it's the simplest to implement, without recognizing the syntactic-fragment cost it imposes on every chunk.
- Reaching for semantic chunking as a default "more advanced = better" choice without confirming sentence-aware chunking is actually the bottleneck — paying ongoing embedding cost for a problem that may not exist in this specific corpus.
- Applying generic sentence-aware chunking to clearly structured Q&A content instead of exploiting the existing structure directly, producing chunks that split a question from its own answer simply because they landed on opposite sides of a size-based cut.
- Assuming one chunking strategy must be used uniformly across the entire corpus, rather than choosing per source type the same way loader granularity was chosen per source type in earlier topics.

### Lead-Level Interview Questions

**Q: Sentence-aware chunking already avoids breaking sentences. Why would semantic chunking ever be worth the extra cost?**
A: Sentence-aware chunking guarantees syntactic integrity, not topical integrity — three unrelated sentences can still legally end up in the same chunk just because they fit the size budget together, producing a blurry, multi-topic embedding. Semantic chunking cuts on meaning shift instead of size, directly targeting topical coherence — worth the extra embedding-at-chunk-time cost specifically when retrieval quality issues are traced back to topically-mixed chunks, not as a default upgrade.

**Q: You have a clean FAQ document with consistent Q&A markers. Why not just run the standard sentence-aware chunker over it like everything else?**
A: Because the document already tells you exactly where the ideal chunk boundaries are — each Q&A pair is a complete, self-contained unit. Running a size-based chunker over it risks cutting a question away from its own answer purely because of where the size limit happened to land, which a structure-aware approach that splits directly on the Q&A markers avoids entirely, with less computational cost than even sentence-aware chunking since no sentence-boundary detection is needed.

**Q: What's the actual failure mode of structure-aware chunking, and how would you detect it in production?**
A: It only works as well as the structure-detection logic — inconsistent formatting in the source document (some Q&A pairs marked clearly, others not) causes silent extraction failures: pairs get missed or merged incorrectly, with no exception thrown. Detecting this requires sampling the actual extracted chunks against the source document and checking extraction completeness, the same "validate the output, don't trust no errors thrown" discipline that applies everywhere else in this pipeline.

**Q: How would you decide which chunking strategy to use for a brand-new source type added to this project?**
A: Start by asking whether the source already has reliable internal structure to exploit (structure-aware first choice if yes). If not, default to sentence-aware as the safe general-purpose baseline. Only escalate to semantic chunking if sentence-aware chunking is measured, not assumed, to be producing topically-incoherent chunks that hurt retrieval — escalating cost should follow evidence of a real problem, not a desire to use a more sophisticated technique.

### Hidden Concepts Worth Knowing

- **The cost-vs-quality ladder repeats across this entire sub-chapter**: fixed-size (cheapest, lowest quality) → sentence-aware (cheap, better) → semantic (expensive, best general-purpose quality) → structure-aware (cheapest of all when applicable, often the *highest* quality because it uses ground-truth structure instead of inferring it) — the same cost-ordering principle seen earlier with hash-before-embedding dedup applies here too: use the cheapest strategy that's actually sufficient for the content.
- **Hybrid strategies are common in real systems**: structure-aware extraction first (split on Q&A markers, SOP steps), falling back to sentence-aware chunking only for any remaining unstructured prose in the same document — not a strict either/or choice per document.

### Revision Summary

> Four chunking strategies trade cost against quality differently: fixed-size is cheap but breaks sentences; sentence-aware (this project's default) preserves sentences but not topical coherence; semantic chunking cuts on meaning-shift for better coherence at real embedding cost; structure-aware chunking exploits a document's own existing structure (Q&A pairs) for the best result at the lowest cost, but only where that structure reliably exists. Choose per source type, escalate to more expensive strategies only on evidence of a real problem.

In [2]:
"""
chunking_strategies.py
------------------------
Side-by-side implementations of four chunking strategies, so they can be
compared directly on the same input text.

  fixed_size_chunk     -- cheapest, lowest quality: cuts every N characters,
                           no regard for word/sentence boundaries.
  sentence_aware_chunk -- this project's default (see chunker.py): never
                           breaks a sentence, no awareness of topic shifts.
  semantic_chunk        -- cuts where consecutive-sentence similarity drops
                           below a threshold, i.e. where the topic changes.
  qa_structure_chunk    -- exploits existing "Q: ... A: ..." structure
                           directly; no size or meaning-based logic at all.
"""

import re
import numpy as np
from document import make_document
from chunker import split_sentences  # reuse the sentence splitter from Topic 2

_model = None


def _get_model():
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        _model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    return _model


# ----------------------------------------------------------------------
# 1. Fixed-size -- cheapest, lowest quality
# ----------------------------------------------------------------------
def fixed_size_chunk(text: str, chunk_size: int = 120) -> list:
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]


# ----------------------------------------------------------------------
# 2. Sentence-aware -- this project's default (no overlap shown here;
#    see chunker.py for the overlap-enabled version)
# ----------------------------------------------------------------------
def sentence_aware_chunk(text: str, chunk_size: int = 120) -> list:
    sentences = split_sentences(text)
    chunks, current, current_len = [], [], 0
    for sentence in sentences:
        if current_len + len(sentence) > chunk_size and current:
            chunks.append(" ".join(current))
            current, current_len = [], 0
        current.append(sentence)
        current_len += len(sentence)
    if current:
        chunks.append(" ".join(current))
    return chunks


# ----------------------------------------------------------------------
# 3. Semantic -- cut where meaning shifts, not where size runs out
# ----------------------------------------------------------------------
def semantic_chunk(text: str, similarity_threshold: float = 0.5) -> list:
    sentences = split_sentences(text)
    if len(sentences) <= 1:
        return sentences

    model = _get_model()
    embeddings = model.encode(sentences, normalize_embeddings=True, show_progress_bar=False)

    chunks, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        similarity = float(np.dot(embeddings[i - 1], embeddings[i]))  # normalized -> dot == cosine
        if similarity < similarity_threshold:
            chunks.append(" ".join(current))
            current = []
        current.append(sentences[i])
    if current:
        chunks.append(" ".join(current))
    return chunks


# ----------------------------------------------------------------------
# 4. Structure-aware -- exploit existing Q&A markers directly
# ----------------------------------------------------------------------
QA_PATTERN = re.compile(r"Q:\s*(.+?)\s*A:\s*(.+?)(?=Q:|$)", re.DOTALL)


def qa_structure_chunk(text: str) -> list:
    """Returns one chunk per Q&A pair, formatted as 'Q: ... A: ...'."""
    pairs = QA_PATTERN.findall(text)
    return [f"Q: {q.strip()} A: {a.strip()}" for q, a in pairs]


if __name__ == "__main__":
    sample = (
        "Premature withdrawal incurs a 1 percent penalty on the applicable rate. "
        "This does not apply if the FD is closed due to the death of the depositor. "
        "Senior citizens receive an additional 0.5 percent interest on all tenures."
    )

    print("--- Fixed-size ---")
    for c in fixed_size_chunk(sample, chunk_size=60):
        print(f"  {c!r}")

    print("\n--- Sentence-aware ---")
    for c in sentence_aware_chunk(sample, chunk_size=120):
        print(f"  {c!r}")

    print("\n--- Semantic ---")
    for c in semantic_chunk(sample, similarity_threshold=0.3):
        print(f"  {c!r}")

    qa_sample = (
        "Q: What is the penalty for premature withdrawal? "
        "A: A 1 percent penalty on the applicable rate. "
        "Q: Do senior citizens get a higher rate? "
        "A: Yes, an additional 0.5 percent on all tenures."
    )
    print("\n--- Structure-aware (Q&A) ---")
    for c in qa_structure_chunk(qa_sample):
        print(f"  {c!r}")

--- Fixed-size ---
  'Premature withdrawal incurs a 1 percent penalty on the appli'
  'cable rate. This does not apply if the FD is closed due to t'
  'he death of the depositor. Senior citizens receive an additi'
  'onal 0.5 percent interest on all tenures.'

--- Sentence-aware ---
  'Premature withdrawal incurs a 1 percent penalty on the applicable rate.'
  'This does not apply if the FD is closed due to the death of the depositor.'
  'Senior citizens receive an additional 0.5 percent interest on all tenures.'

--- Semantic ---

  'Premature withdrawal incurs a 1 percent penalty on the applicable rate.'
  'This does not apply if the FD is closed due to the death of the depositor.'
  'Senior citizens receive an additional 0.5 percent interest on all tenures.'

--- Structure-aware (Q&A) ---
  'Q: What is the penalty for premature withdrawal? A: A 1 percent penalty on the applicable rate.'
  'Q: Do senior citizens get a higher rate? A: Yes, an additional 0.5 percent on all tenures.'
